In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t
from pathlib import Path

# ============================================================
# 0. GENERAL SETTINGS
# ============================================================

N_VALUES = [20, 50, 100, 200, 1000]
K = 1000
CONFIDENCE_LEVEL = 0.95
ALPHA = 1 - CONFIDENCE_LEVEL

OUTPUT_DIR = Path("simulation_results")
OUTPUT_DIR.mkdir(exist_ok=True)

rng = np.random.default_rng(42)

# Set to False if you only want to save the plots as PNG files.
SHOW_PLOTS = True


# ============================================================
# 1. GENERATE K SAMPLES, EACH CONTAINING n OBSERVATIONS
# ============================================================

def generate_samples(distribution, n, **params):
    """
    Generate K independent experiments.

    Returns an array of shape (K, n):
    each row represents one experiment.
    """

    if distribution == "Gaussian":
        return rng.normal(
            loc=params["mu"],
            scale=params["sigma"],
            size=(K, n)
        )

    elif distribution == "Uniform":
        return rng.uniform(
            low=params["low"],
            high=params["high"],
            size=(K, n)
        )

    elif distribution == "Bernoulli":
        return rng.binomial(
            n=1,
            p=params["p"],
            size=(K, n)
        )

    else:
        raise ValueError("Unsupported distribution")


# ============================================================
# 2. RUN ONE SET OF K EXPERIMENTS AND CALCULATE CIs
# ============================================================

def run_experiment(
    distribution,
    true_mu,
    true_sigma,
    n,
    sigma_known,
    params,
    section
):
    """
    Calculate sample means, 95% confidence intervals,
    coverage count, and plots for one configuration.
    """

    # Step 1: Generate K samples of size n
    samples = generate_samples(distribution, n, **params)

    # Step 2: Calculate one sample mean for each experiment
    sample_means = np.mean(samples, axis=1)

    # Step 3: Calculate standard errors and critical values
    if sigma_known:
        # Part (a), or known-sigma cases in Parts (c) and (d)
        standard_error = true_sigma / np.sqrt(n)
        critical_value = norm.ppf(1 - ALPHA / 2)

        method = "Known sigma: z interval"

    else:
        # Part (b): estimate sigma from each sample
        sample_std = np.std(samples, axis=1, ddof=1)
        standard_error = sample_std / np.sqrt(n)
        critical_value = t.ppf(1 - ALPHA / 2, df=n - 1)

        method = "Unknown sigma: t interval"

    # Step 4: Calculate the confidence interval limits
    lower = sample_means - critical_value * standard_error
    upper = sample_means + critical_value * standard_error

    # Step 5: Check whether each interval contains the true mean
    contains_mean = (lower <= true_mu) & (true_mu <= upper)

    coverage_count = int(np.sum(contains_mean))
    coverage_percentage = 100 * coverage_count / K

    # Step 6: Plot the histogram of the sample means
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(
        sample_means,
        bins=30,
        edgecolor="black",
        alpha=0.75
    )

    axes[0].axvline(
        true_mu,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"True mean = {true_mu:.4f}"
    )

    axes[0].set_title(
        f"Histogram of Sample Means\n"
        f"{distribution}, n={n}, K={K}"
    )
    axes[0].set_xlabel("Sample mean")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    # Step 7: Plot lower and upper confidence limits
    experiment_numbers = np.arange(1, K + 1)

    axes[1].scatter(
        experiment_numbers,
        lower,
        color="blue",
        s=9,
        label="Lower limit"
    )

    axes[1].scatter(
        experiment_numbers,
        upper,
        color="red",
        s=9,
        label="Upper limit"
    )

    axes[1].axhline(
        true_mu,
        color="green",
        linestyle="--",
        linewidth=1.5,
        label=f"True mean = {true_mu:.4f}"
    )

    axes[1].set_title(
        f"95% Confidence Interval Limits\n"
        f"{distribution}, n={n}, K={K}"
    )
    axes[1].set_xlabel("Experiment number")
    axes[1].set_ylabel("Confidence interval limit")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    # Explicitly identify the population sigma and method.
    # In Part (b), sigma is shown for evaluation, but the
    # confidence interval calculation does not use it.
    sigma_description = (
        f"Population sigma = {true_sigma:.4f} (known)"
        if sigma_known
        else f"Population sigma = {true_sigma:.4f} (unknown; estimated by s)"
    )

    fig.suptitle(
        f"{section}: {method}\n"
        f"{sigma_description}; true mean = {true_mu:.4f}",
        fontsize=12
    )

    fig.tight_layout(rect=[0, 0, 1, 0.88])

    # Use a unique filename for every configuration.
    filename = (
        f"{section.replace(' ', '_')}_"
        f"{distribution}_n{n}_"
        f"sigma{true_sigma:.4f}_"
        f"{'known' if sigma_known else 'unknown'}.png"
    )

    fig.savefig(OUTPUT_DIR / filename, dpi=150)

    if SHOW_PLOTS:
        plt.show()

    plt.close(fig)

    # Step 8: Return the experiment results
    return {
        "section": section,
        "distribution": distribution,
        "method": method,
        "n": n,
        "true_mu": true_mu,
        "true_sigma": true_sigma,
        "coverage_count": coverage_count,
        "coverage_percentage": coverage_percentage,
        "mean_of_sample_means": float(np.mean(sample_means)),
        "average_interval_width": float(np.mean(upper - lower))
    }


# ============================================================
# 3. PRINT THE RESULTS FOR EACH SECTION SEPARATELY
# ============================================================

all_results = []


def print_section_results(section, results):
    print("\n" + "=" * 112)
    print(section)
    print("=" * 112)

    print(
        f"{'Distribution':<13}"
        f"{'Method':<25}"
        f"{'n':>7}"
        f"{'True mu':>12}"
        f"{'True sigma':>13}"
        f"{'Covered':>11}"
        f"{'Coverage %':>13}"
        f"{'Avg width':>13}"
    )

    print("-" * 112)

    for r in results:
        print(
            f"{r['distribution']:<13}"
            f"{r['method']:<25}"
            f"{r['n']:>7}"
            f"{r['true_mu']:>12.4f}"
            f"{r['true_sigma']:>13.4f}"
            f"{r['coverage_count']:>11}"
            f"{r['coverage_percentage']:>12.2f}%"
            f"{r['average_interval_width']:>13.4f}"
        )


# ============================================================
# PART (a): GAUSSIAN, KNOWN POPULATION STANDARD DEVIATION
# mu = 0, sigma = 2, 3, 5
# ============================================================

part_a_results = []

for sigma in [2, 3, 5]:
    for n in N_VALUES:
        result = run_experiment(
            distribution="Gaussian",
            true_mu=0,
            true_sigma=sigma,
            n=n,
            sigma_known=True,
            params={"mu": 0, "sigma": sigma},
            section="Part A - Known Sigma"
        )

        part_a_results.append(result)

all_results.extend(part_a_results)
print_section_results("PART (a): GAUSSIAN WITH KNOWN SIGMA", part_a_results)


# ============================================================
# PART (b): GAUSSIAN, UNKNOWN POPULATION STANDARD DEVIATION
# The same distributions and sample sizes as Part (a)
# ============================================================

part_b_results = []

for sigma in [2, 3, 5]:
    for n in N_VALUES:
        result = run_experiment(
            distribution="Gaussian",
            true_mu=0,
            true_sigma=sigma,
            n=n,
            sigma_known=False,
            params={"mu": 0, "sigma": sigma},
            section="Part B - Unknown Sigma"
        )

        part_b_results.append(result)

all_results.extend(part_b_results)
print_section_results("PART (b): GAUSSIAN WITH UNKNOWN SIGMA", part_b_results)


# ============================================================
# PART (c): UNIFORM DISTRIBUTIONS
# Uniform(a, b): mean = (a+b)/2
#                variance = (b-a)^2/12
# ============================================================

part_c_results = []

for low, high in [(0, 1), (-1, 1), (-2, 2)]:

    true_mu = (low + high) / 2
    true_sigma = (high - low) / np.sqrt(12)

    for n in N_VALUES:
        result = run_experiment(
            distribution="Uniform",
            true_mu=true_mu,
            true_sigma=true_sigma,
            n=n,
            sigma_known=True,
            params={"low": low, "high": high},
            section="Part C - Uniform Known Sigma"
        )

        part_c_results.append(result)

all_results.extend(part_c_results)
print_section_results("PART (c): UNIFORM DISTRIBUTIONS", part_c_results)


# ============================================================
# PART (d): BERNOULLI DISTRIBUTION
# Assume p = 0.5 because the question does not specify p.
# Mean = p; standard deviation = sqrt(p * (1-p)).
# ============================================================

part_d_results = []

p = 0.5
true_mu = p
true_sigma = np.sqrt(p * (1 - p))

for n in N_VALUES:
    result = run_experiment(
        distribution="Bernoulli",
        true_mu=true_mu,
        true_sigma=true_sigma,
        n=n,
        sigma_known=True,
        params={"p": p},
        section="Part D - Bernoulli Known Sigma"
    )

    part_d_results.append(result)

all_results.extend(part_d_results)
print_section_results("PART (d): BERNOULLI DISTRIBUTION", part_d_results)


# ============================================================
# 4. FINAL SUMMARY
# ============================================================

print("\nAll experiments completed.")
print(f"Total configurations: {len(all_results)}")
print(f"Total confidence intervals: {len(all_results) * K}")
print(f"Plots saved in: {OUTPUT_DIR.resolve()}")
